# LabID: предобработка ветвей для T и P, визуальная проверка

Зачем: после поправки по z (0,7 для Wt и Ab) пригодны для T лишь 44 из 179 ветвей, для P 82. Ноутбук помогает понять, **что именно** не проходит и справедливы ли пороги, подобранные на Minnie65:

1. таблица пригодности по группам и частота причин отказа;
2. **пересчёт числа пригодных ветвей при других порогах** (без пересчёта геометрии, по записанным величинам): какой порог держит больше всего ветвей;
3. 3D-графики: лучшие ветви каждой группы, ветви, отброшенные только по одной причине (`trim`, `cross_section`, `centerline`), ветви без центральной линии, положения шипиков (для P).

Группа **Wt_old** по умолчанию не показывается (после поправки 0,7 она всё ещё растянута по z примерно в 2,4 раза, в анализ не берётся).

Только чтение: ничего не пересчитывается. Сетки ветвей читаются из преобразованного дерева LabID.

In [ ]:
import os
import sys
from pathlib import Path


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "src" / "neuron_model").is_dir():
            return candidate
    raise RuntimeError("Repository root was not found.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go

try:
    pd.set_option("future.no_silent_downcasting", True)
except Exception:
    pass

from src.neuron_model import branch_geometry as bg
from src.neuron_model.branch_preprocessing import load_config

# --- параметры (переменные окружения переопределяют) -------------------------------------------------------------
BRANCHES_ROOT = Path(os.environ.get("BRANCHES_ROOT", r"C:/Users/Student/LipsEkaterina/data/labid/preprocessed_branches_z07"))
RAW_ROOT = Path(os.environ.get("RAW_ROOT", r"C:/Users/Student/LipsEkaterina/data/labid/labid_raw_z07/labid"))   # <RAW_ROOT>/<ветвь>/limb_000/branch_000/branch_mesh.off
CONFIG_PATH = REPO_ROOT / "data/processed/config/branch_preprocessing_labid.yaml"
EXCLUDE_GROUPS = ("Wt_old",)
N_EXAMPLES = 1          # примеров на категорию
SHOW_MESH = True

cfg = load_config(CONFIG_PATH)
print("корень результатов:", BRANCHES_ROOT)
print("пороги пригодности:", {k: getattr(cfg, k) for k in ("invalid_section_fraction_max", "trim_max_nm", "length_min_t_nm", "unknown_fraction_max", "position_fraction_min", "centerline_inside_min", "spline_rms_floor_nm", "long_gap_nm")})

In [ ]:
branches_all = pd.read_parquet(BRANCHES_ROOT / "branches.parquet")
spines_all = pd.read_parquet(BRANCHES_ROOT / "branch_spines.parquet")
branches = branches_all[~branches_all["labid_group"].isin(EXCLUDE_GROUPS)].copy()
ok = branches[branches["status"] == "ok"]
print(f"ветвей {len(branches_all)}; после исключения {EXCLUDE_GROUPS}: {len(branches)}; статус ok: {len(ok)}")
display(branches.groupby("labid_group").agg(ветвей=("status", "size"), ok=("status", lambda s: int((s == "ok").sum())), без_линии=("status", lambda s: int((s == "no_centerline").sum())),
                                            годны_T=("train_eligible_T", "sum"), годны_P=("train_eligible_P", "sum"), радиус_нм=("radius_median_nm", "median"), длина_мкм=("length_nm", lambda s: s.median() / 1000)).round(1))
from collections import Counter

cnt_T, cnt_P = Counter(), Counter()
for _, r in ok.iterrows():
    reasons = [x for x in str(r["qc_reason"] or "").split(";") if x]
    if not r["train_eligible_T"]:
        cnt_T.update(x for x in reasons if x != "spines" and x != "soma")
    if not r["train_eligible_P"]:
        cnt_P.update(x for x in reasons if x in ("centerline", "spines", "radius_range", "merge"))
print("не годны для T, по причинам (soma не блокирует; одна ветвь может не пройти несколько):", dict(cnt_T.most_common()))
print("не годны для P:", dict(cnt_P.most_common()))

## 1. Что изменится при других порогах (без пересчёта геометрии)

Пороги `invalid_section_fraction_max` (доля недостоверных сечений) и `trim_max_nm` (на сколько обрезаны концы) подобраны на Minnie65 (радиус ~ 300 нм, концы — места ветвления). В LabID радиус ~ 740 нм, а концы фрагментов — границы кадра, поэтому абсолютный порог обрезки 2 мкм может быть несправедлив: минимальная обрезка 2 радиуса = 1,5 мкм у каждого конца. Таблица показывает число пригодных для T ветвей, если порог обрезки задать как **доля радиуса** (4, 6, 8 r) или в нм, и при разных долях недостоверных сечений.

In [ ]:
def eligible_T(r, invalid_max, trim_nm=None, trim_factor=None):
    """Пригодность для T по записанным величинам; trim: абсолютный порог (нм) или доля радиуса."""
    if r["status"] != "ok":
        return False
    rmed = r["radius_median_nm"]
    trim_lim = trim_nm if trim_nm is not None else trim_factor * rmed
    return bool(cfg.length_min_t_nm <= r["length_nm"] <= cfg.length_max_t_nm and r["qc_merge"] and r["section_invalid_fraction"] <= invalid_max
                and r["qc_centerline"] and r["qc_frame"] and r["qc_tube"] and cfg.radius_min_nm <= rmed <= cfg.radius_max_nm
                and r["trim_start_nm"] <= trim_lim and r["trim_end_nm"] <= trim_lim)


rows = []
for invalid_max in (0.4, 0.5, 0.6):
    out = {"недостоверных сечений ≤": invalid_max}
    for label, kw in (("trim 2000 нм (сейчас)", dict(trim_nm=2000.0)), ("trim 4000 нм", dict(trim_nm=4000.0)), ("trim 4 r", dict(trim_factor=4.0)), ("trim 6 r", dict(trim_factor=6.0)), ("trim 8 r", dict(trim_factor=8.0))):
        out[label] = int(sum(eligible_T(r, invalid_max, **kw) for _, r in ok.iterrows()))
    rows.append(out)
print(f"число ветвей, пригодных для T (из {len(ok)} ok без {EXCLUDE_GROUPS}):")
display(pd.DataFrame(rows).set_index("недостоверных сечений ≤"))
print("медианы: обрезка начала/конца, нм:", float(ok["trim_start_nm"].median()), float(ok["trim_end_nm"].median()), "; обрезка / радиус:", float((np.maximum(ok["trim_start_nm"], ok["trim_end_nm"]) / ok["radius_median_nm"]).median()))

## Вспомогательные функции

`explain(row)` перечисляет непройденные проверки по текущим порогам, `plot_branch(row)` строит график (сетка ветви, исходный скелет, линия, центры сечений зелёные/красные, основания шипиков).

In [ ]:
FLAG_NAMES = {1: "few_points", 2: "large_radius", 4: "kink", 8: "outside_mesh", 16: "end_section"}


def explain(row) -> str:
    """Причины непригодности ветви: значение > порог."""
    def missing(v):
        return v is None or (isinstance(v, float) and np.isnan(v))

    r = lambda k, d=np.nan: d if missing(row.get(k)) else row.get(k)
    flag = lambda k, d=False: d if missing(row.get(k)) else bool(row.get(k))
    out = []
    if row["status"] == "too_short":
        return f"too_short: длина по таблице {r('table_length_nm'):.0f} нм < {cfg.length_min_p_nm:.0f} нм (геометрия не считалась)"
    if row["status"] == "no_centerline":
        return (f"no_centerline: надёжных сечений {int(r('n_reliable_sections', 0))} из {int(r('n_sections', 0))} < {cfg.min_reliable_sections} "
                f"(длина по таблице {r('table_length_nm'):.0f} нм)")
    if row["status"] != "ok":
        return f"{row['status']}: {row.get('error')}"
    rmed = r("radius_median_nm")
    T, P = [], []
    L = r("length_nm")
    if not (cfg.length_min_t_nm <= L <= cfg.length_max_t_nm):
        T.append(f"длина {L:.0f} нм вне [{cfg.length_min_t_nm:.0f}; {cfg.length_max_t_nm:.0f}]")
    if r("merge_max_gap_nm", 0.0) > cfg.join_distance_nm or not flag("skeleton_topology_ok", True):
        T.append(f"merge: разрыв между членами {r('merge_max_gap_nm', 0):.0f} нм (порог {cfg.join_distance_nm:.0f}) / топология скелета ok: {flag('skeleton_topology_ok', True)}")
    if r("section_invalid_fraction", 1.0) > cfg.invalid_section_fraction_max:
        T.append(f"cross_section: недостоверных сечений {r('section_invalid_fraction', 1.0):.2f} > {cfg.invalid_section_fraction_max}")
    lim = max(cfg.spline_rms_factor * rmed, cfg.spline_rms_floor_nm)
    gap_bad = r("longest_gap_nm", np.inf) > cfg.long_gap_nm
    rms_bad = r("centerline_rms_to_centers_nm", np.inf) > lim
    if gap_bad or rms_bad:
        s_ = []
        if rms_bad:
            s_.append(f"отклонение линии от центров {r('centerline_rms_to_centers_nm'):.0f} нм > {lim:.0f} нм (= max({cfg.spline_rms_factor}·r_med {rmed:.0f}, пол {cfg.spline_rms_floor_nm:.0f}))")
        if gap_bad:
            s_.append(f"наибольший пропуск линии {r('longest_gap_nm'):.0f} нм > {cfg.long_gap_nm:.0f}")
        T.append("centerline: " + "; ".join(s_))
        P.append("centerline")
    if r("twist_max_per_nm", np.inf) > cfg.twist_tolerance_per_nm:
        T.append(f"frame: кручение {r('twist_max_per_nm'):.1e} > {cfg.twist_tolerance_per_nm:.0e}")
    if flag("tube_self_contact", True) or r("max_r_kappa", np.inf) >= 1.0:
        T.append(f"tube: самоконтакт {flag('tube_self_contact', True)}, max r·κ {r('max_r_kappa', np.inf):.2f}")
    if not (cfg.radius_min_nm <= rmed <= cfg.radius_max_nm):
        T.append(f"радиус {rmed:.0f} нм вне [{cfg.radius_min_nm:.0f}; {cfg.radius_max_nm:.0f}]"); P.append("радиус")
    if r("trim_start_nm", np.inf) > cfg.trim_max_nm or r("trim_end_nm", np.inf) > cfg.trim_max_nm:
        T.append(f"trim: обрезка концов {r('trim_start_nm'):.0f} / {r('trim_end_nm'):.0f} нм > {cfg.trim_max_nm:.0f}")
    in_range = r("n_spines_in_range", 0)
    if r("unknown_fraction", 1.0) > cfg.unknown_fraction_max:
        P.append(f"spines: сегментов без вердикта {r('unknown_fraction', 1.0):.2f} > {cfg.unknown_fraction_max}")
    if in_range and r("n_with_position", 0) < cfg.position_fraction_min * in_range:
        P.append(f"spines: положение определено у {int(r('n_with_position', 0))} из {int(in_range)} шипиков (нужно >= {cfg.position_fraction_min:.0%})")
    if flag("train_eligible_T") and flag("train_eligible_P"):
        return "годна для T и P"
    note = " (флаги в таблице записаны при другом наборе порогов: выполните reapply_eligibility.py)"
    parts = []
    if not flag("train_eligible_T"):
        parts.append("T: " + " | ".join(T) if T else "T: по текущим порогам причин нет" + note)
    if not flag("train_eligible_P"):
        parts.append("P: " + " | ".join(P) if P else "P: по текущим порогам причин нет" + note)
    return "  ||  ".join(parts)


def load_geometry(row):
    path = BRANCHES_ROOT / str(row["neuron_id"]) / f"limb_{int(row['limb_id']):03d}" / f"merged_{int(row['merged_branch_id'])}"
    out = {"sections": None, "geometry": None}
    if (path / "branch_sections.npz").exists():
        out["sections"] = dict(np.load(path / "branch_sections.npz"))
    if (path / "branch_geometry.npz").exists():
        out["geometry"] = dict(np.load(path / "branch_geometry.npz"))
    return out


def load_mesh(row):
    import trimesh

    members = [int(x) for x in str(row.get("member_branch_ids", "")).split("|") if x]
    vs, fs, offset = [], [], 0
    for b in members:
        p = RAW_ROOT / str(row["neuron_id"]) / f"limb_{int(row['limb_id']):03d}" / f"branch_{b:03d}" / "branch_mesh.off"
        m = trimesh.load(p, process=False)
        vs.append(np.asarray(m.vertices, float)); fs.append(np.asarray(m.faces) + offset); offset += len(m.vertices)
    return (np.vstack(vs), np.vstack(fs)) if vs else (None, None)


def folders(row):
    """Имена папок в данных: limb_001/branch_10035 (+ другие члены объединённой ветви)."""
    members = [int(x) for x in str(row.get("member_branch_ids", "")).split("|") if x]
    limb = f"limb_{int(row['limb_id']):03d}"
    names = ", ".join(f"branch_{b:03d}" for b in members) if members else f"branch_{int(row['merged_branch_id']):03d}"
    return f"{row.get('labid_group')} / {row['neuron_id']} / {limb}/{names}  (объединённая ветвь {int(row['merged_branch_id'])}" + (f", членов: {len(members)})" if len(members) > 1 else ")")


def spine_counts(row):
    sp = spines_all[(spines_all["neuron_id"] == row["neuron_id"]) & (spines_all["limb_id"] == row["limb_id"]) & (spines_all["merged_branch_id"] == row["merged_branch_id"])]
    if not len(sp):
        return "сегментов шипиков в папках нет"
    st = sp["status"].value_counts().to_dict()
    pos = int(sp["has_position"].fillna(False).sum())
    return (f"сегментов в папках spines/: {len(sp)} (настоящих шипиков {st.get('valid', 0)}, ложных — части ствола {st.get('invalid', 0)}, без вердикта {st.get('unknown', 0)}); "
            f"с определённым положением: {pos}")


def section_flag_counts(row):
    n = row.get("n_sections")
    if n is None or (isinstance(n, float) and np.isnan(n)):
        return ""
    parts = [f"{name} {int(row.get('sections_' + name) or 0)}" for name in FLAG_NAMES.values() if f"sections_{name}" in row]
    return f"сечений {int(n)}, достоверных {int(row.get('n_reliable_sections') or 0)}; флаги (сечение может нести несколько): " + ", ".join(parts)


def flag_text(f):
    names = [n for bit, n in FLAG_NAMES.items() if int(f) & bit]
    return ", ".join(names) if names else "достоверное"


def plot_branch(row, show_mesh=SHOW_MESH, title_extra=""):
    g = load_geometry(row)
    sec, geo = g["sections"], g["geometry"]
    fig = go.Figure()
    if show_mesh:
        try:
            v, f = load_mesh(row)
            fig.add_trace(go.Mesh3d(x=v[:, 0], y=v[:, 1], z=v[:, 2], i=f[:, 0], j=f[:, 1], k=f[:, 2], color="lightgrey", opacity=0.25, name="сетка ветви (с шипиками)", flatshading=True))
        except Exception as exc:
            print("сетка недоступна:", repr(exc)[:150])
    if geo is not None:
        sk = geo["skeleton"]
        fig.add_trace(go.Scatter3d(x=sk[:, 0], y=sk[:, 1], z=sk[:, 2], mode="lines", line=dict(color="black", width=2, dash="dot"), name="исходный скелет (на мембране)"))
        gm = geo["gamma"]
        fig.add_trace(go.Scatter3d(x=gm[:, 0], y=gm[:, 1], z=gm[:, 2], mode="lines+markers", marker=dict(size=2), line=dict(color="royalblue", width=6), name="центральная линия (сплайн), узлы 500 нм"))
        v_, f_ = bg.tube_mesh(gm, geo["radius"], geo["frame_e1"], geo["frame_e2"], n_theta=24, caps=False)
        fig.add_trace(go.Mesh3d(x=v_[:, 0], y=v_[:, 1], z=v_[:, 2], i=f_[:, 0], j=f_[:, 1], k=f_[:, 2], color="royalblue", opacity=0.3, name="трубка по радиусу (круг)", visible="legendonly"))
        if "ellipse_a" in geo:
            ve, fe = bg.ellipse_tube_mesh(gm, geo["ellipse_a"], geo["ellipse_b"], geo["ellipse_angle"], geo["frame_e1"], geo["frame_e2"], n_theta=24)
            fig.add_trace(go.Mesh3d(x=ve[:, 0], y=ve[:, 1], z=ve[:, 2], i=fe[:, 0], j=fe[:, 1], k=fe[:, 2], color="darkorange", opacity=0.3,
                                    name=f"трубка-эллипс (b/a медиана {np.median(geo['ellipse_b'] / geo['ellipse_a']):.2f})", visible="legendonly"))
    elif sec is not None:
        p = sec["points"]
        fig.add_trace(go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode="lines", line=dict(color="black", width=2, dash="dot"), name="исходный скелет (перепробован)"))
    if sec is not None:
        c, fl, rad = sec["centers"], sec["flags"], sec["radii"]
        fit = np.isfinite(rad)
        good = fit & (fl == 0)
        bad = fit & (fl != 0)
        fig.add_trace(go.Scatter3d(x=c[good, 0], y=c[good, 1], z=c[good, 2], mode="markers", marker=dict(size=4, color="green"), name=f"центры сечений: достоверные ({int(good.sum())})",
                                   text=[f"r {r_:.0f} нм" for r_ in rad[good]], hoverinfo="text"))
        fig.add_trace(go.Scatter3d(x=c[bad, 0], y=c[bad, 1], z=c[bad, 2], mode="markers", marker=dict(size=5, color="red", symbol="diamond"), name=f"центры сечений: с флагом ({int(bad.sum())})",
                                   text=[f"{flag_text(f_)}; r {r_:.0f} нм" for f_, r_ in zip(fl[bad], rad[bad])], hoverinfo="text"))
    if geo is not None and "s" in spines_all.columns:  # no spine of the neuron has a position -> no column s
        sp = spines_all[(spines_all["neuron_id"] == row["neuron_id"]) & (spines_all["limb_id"] == row["limb_id"]) & (spines_all["merged_branch_id"] == row["merged_branch_id"]) & (spines_all["status"] == "valid") & spines_all["s"].notna()]
        if len(sp):
            s_, th = sp["s"].to_numpy(float), sp["theta"].to_numpy(float)
            interp = lambda a: np.stack([np.interp(s_, geo["s"], a[:, k]) for k in range(3)], 1)
            r_ = np.interp(s_, geo["s"], geo["radius"])
            anchor = interp(geo["gamma"]) + r_[:, None] * (np.cos(th)[:, None] * interp(geo["frame_e1"]) + np.sin(th)[:, None] * interp(geo["frame_e2"]))
            kind = np.where(sp["projects_to_end"].fillna(False), "за концом линии", np.where(sp["position_outlier"].fillna(False), "выброс по расстоянию", "положение определено"))
            for name, color in (("положение определено", "magenta"), ("выброс по расстоянию", "orange"), ("за концом линии", "red")):
                m = kind == name
                if m.any():
                    fig.add_trace(go.Scatter3d(x=anchor[m, 0], y=anchor[m, 1], z=anchor[m, 2], mode="markers", marker=dict(size=5, color=color, symbol="circle"), name=f"основания шипиков: {name} ({int(m.sum())})",
                                               text=[f"u {u:.2f}, θ {t:.2f}, |d-r| {abs(d):.0f} нм" for u, t, d in zip(sp["u"][m], sp["theta"][m], sp["dist_perp_minus_radius"][m])], hoverinfo="text"))
    head = f"{folders(row)}: длина {row.get('length_nm', row.get('table_length_nm', np.nan)) / 1000:.1f} мкм, r_med {row.get('radius_median_nm', np.nan):.0f} нм"
    sub = spine_counts(row)
    fig.update_layout(title=dict(text=head + "<br><sup>" + sub + (" | " + title_extra if title_extra else "") + "</sup>", font=dict(size=13)), scene=dict(aspectmode="data"), height=720, legend=dict(font=dict(size=10)))
    return fig


def show_examples(frame, n, header):
    print(header)
    for _, row in frame.head(n).iterrows():
        print(f"\n{folders(row)}:", explain(row.to_dict()))
        print("   ", section_flag_counts(row.to_dict()))
        plot_branch(row.to_dict()).show()


## 2. Лучшие ветви каждой группы (годны для T)

Что искать: линия идёт по оси сетки, почти все центры зелёные, основания шипиков лежат на поверхности ствола. Если даже здесь много красных центров, дело в самих сетках (конфокальное размытие), а не в порогах.

In [ ]:
for g in ("Wt", "Ab", "Wt+9009", "Ab+9009"):
    sel = ok[(ok["labid_group"] == g) & ok["train_eligible_T"]].sort_values("section_invalid_fraction")
    show_examples(sel, N_EXAMPLES, f"=== {g}: годных для T {len(sel)}; показаны с наименьшей долей недостоверных сечений ===")

## 3. Ветви, отброшенные только по одной причине

Для каждой причины показана ветвь, которая не проходит **только по ней** (если такие есть). Вопрос: действительно ли ветвь плохая или порог слишком строгий (например, обрезка `trim`, когда линия выглядит хорошо).

In [ ]:
bad_T = ok[~ok["train_eligible_T"]]
for reason in ("trim", "cross_section", "centerline", "radius_range", "tube"):
    only = bad_T[bad_T["qc_reason"].fillna("").str.replace("soma;?", "", regex=True).str.strip(";") == reason]
    pool = only if len(only) else bad_T[bad_T["qc_reason"].fillna("").str.contains(reason)]
    show_examples(pool.sort_values("length_nm", ascending=False), N_EXAMPLES, f"=== причина: {reason}; ветвей {len(pool)} (только по ней: {len(only)}) ===")

## 4. Ветви без центральной линии (`no_centerline`)

Меньше пяти достоверных сечений. Обычно очень короткие или с неразделённым ветвлением; для LabID проверить, нет ли среди них длинных ветвей, где сечения ошибочно не подобрались.

In [ ]:
nc = branches[branches["status"] == "no_centerline"].sort_values("table_length_nm", ascending=False)
print(f"no_centerline: {len(nc)} ветвей (без {EXCLUDE_GROUPS})")
if len(nc):
    display(nc[["labid_group", "neuron_id", "table_length_nm", "radius_median_nm", "n_sections", "n_reliable_sections"]].head(15))
show_examples(nc, N_EXAMPLES + 1, "=== no_centerline: самые длинные ===")

## 5. Для P: положения шипиков

Положение шипика определено у 76 % шипиков. Основания шипиков, помеченные оранжевым («выброс по расстоянию»), лежат дальше от поверхности ствола, чем допускает правило (глубже 0,5 r внутри или дальше max(2 r, 400 нм) снаружи). Ниже ветви с наибольшей долей шипиков без положения.

In [ ]:
sp = spines_all.copy()
sp["no_pos"] = ~sp["has_position"].fillna(False).astype(bool)
share = sp.groupby(["neuron_id", "limb_id", "merged_branch_id"]).agg(n=("no_pos", "size"), без_положения=("no_pos", "sum")).reset_index()
share["доля"] = share["без_положения"] / share["n"]
cand = share[(share["n"] >= 5)].merge(ok, on=["neuron_id", "limb_id", "merged_branch_id"]).sort_values("доля", ascending=False)
display(cand[["labid_group", "neuron_id", "n", "без_положения", "доля", "radius_median_nm"]].head(10))
print("расстояние основания от поверхности ствола d - r, нм (квантили 10/50/90):", np.nanpercentile(sp["dist_perp_minus_radius"], [10, 50, 90]).round(0), " относительно r:",
      np.nanpercentile(sp["dist_perp_minus_radius"].to_numpy() / sp["radius_at_spine_nm"].to_numpy() if "radius_at_spine_nm" in sp else sp["dist_perp_minus_radius"] / 740.0, [10, 50, 90]).round(2))
show_examples(cand, N_EXAMPLES, "=== ветви с наибольшей долей шипиков без положения ===")